# TF-Mamba NTU-Fi — Kaggle Lite

**Model:** TF-Mamba baseline — NTU-Fi dataset (paper reproduction)  
**Dataset:** NTU-Fi — 6 activities (box, circle, clean, fall, run, walk), pre-split train_amp/test_amp  
**Preprocessing:** loadmat CSIamp → (x-42.3199)/4.9802 → downsample 2000→500 → reshape (3,114,500) → flatten (342,500) → DWT → XH(250,171), XV(171,250)  
**Notebook:** Imports code from a Kaggle dataset; no inline code definitions.

---

## Required attached datasets

| Dataset name | Role | Mount path |
|---|---|---|
| `4-baselines-code` | Project source code | `/kaggle/input/datasets/imhoangt/4-baselines-code/` |
| `ntu_fi_dataset` | NTU-Fi .mat files | `/kaggle/input/datasets/imhoangt/ntu_fi_dataset/` |

---

## Hyperparameters

| Parameter | Value | Note |
|---|---|---|
| Epochs | 40 max | Early stop when loss < 0.01 |
| LR | 1e-4 | |
| Batch | 32 | |
| Seed | 42 | |
| Split | Pre-existing train_amp/test_amp | |

> This is the original paper reproduction — **do not change hyperparameters**.

In [ ]:
# Cell 1 — Install mamba-ssm
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
print('Install done')

In [ ]:
# Cell 2 — Mount code dataset
import sys
from pathlib import Path

# Detect layout: flat (baselines/ at root) or nested (har_csi/baselines/)
_base = Path('/kaggle/input/datasets/imhoangt/4-baselines-code')
if (_base / 'baselines').exists():
    CODE_PATH = _base
elif (_base / 'har_csi' / 'baselines').exists():
    CODE_PATH = _base / 'har_csi'
else:
    raise FileNotFoundError(
        f"Cannot find 'baselines/' under {_base}.\n"
        "Ensure the 4-baselines-code dataset contains the project files."
    )

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

# Quick import check
from baselines.tf_mamba.tf_mamba_original.tf_mamba_ntu_fi.trainer import run
print('Import OK  : baselines.tf_mamba.tf_mamba_original.tf_mamba_ntu_fi.trainer.run')

In [ ]:
# Cell 3 — Configuration
DATA_ROOT  = Path('/kaggle/input/datasets/imhoangt/ntu_fi_dataset/NTU-Fi')
OUTPUT_DIR = Path('/kaggle/working/outputs/runs/tf_mamba_ntufi')

print(f'Data root  : {DATA_ROOT}')
print(f'Output dir : {OUTPUT_DIR}')

# Check data directories
for dname in ['train_amp', 'test_amp']:
    p = DATA_ROOT / dname
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p}')

In [ ]:
# Cell 4 — Run training
# Note: NTU-Fi loads all .mat files at startup — may take a few minutes.
# Early stop when train loss < 0.01.
run(data_root=DATA_ROOT, output_dir=OUTPUT_DIR)

In [ ]:
# Cell 5 — Results
import json

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print(f"\n{'='*55}")
    print(f"  TF-Mamba NTU-Fi")
    print(f"  Accuracy   : {m['test_accuracy']*100:.2f}%")
    print(f"  F1 Macro   : {m['test_f1_macro']*100:.2f}%")
    print(f"  Params     : {m['params_M']}M")
    print(f"  Stopped ep : {m['stopped_epoch']}")
    if m.get('macs_G'):
        print(f"  MACs       : {m['macs_G']:.3f}G")
    if m.get('latency_mean_ms') is not None:
        print(f"  Latency    : {m['latency_mean_ms']:.2f} +- {m['latency_std_ms']:.2f} ms")
    print(f"  Time       : {m['total_time_s']}s")
    print(f"{'='*55}")
    if m.get('test_f1_per_class'):
        print(f"\n  Per-class F1:")
        classes = ['box', 'circle', 'clean', 'fall', 'run', 'walk']
        for i, v in enumerate(m['test_f1_per_class']):
            print(f"    {classes[i]:8s}: {v*100:.2f}%")
else:
    print('metrics.json not found — training may not have completed.')

In [ ]:
# Cell 6 — Plots and ZIP archive
import csv, json, io, zipfile
import numpy as np
import matplotlib.pyplot as plt

_mpath = OUTPUT_DIR / 'metrics.json'
_lpath = OUTPUT_DIR / 'training_log.csv'
if not _mpath.exists() or not _lpath.exists():
    print('Insufficient data - run Cell 4 first.')
else:
    with open(_mpath) as f:
        m = json.load(f)
    log = []
    with open(_lpath, newline='') as f:
        for row in csv.DictReader(f):
            log.append({k: float(v) for k, v in row.items()})

    epochs   = [r['epoch']        for r in log]
    losses   = [r['train_loss']   for r in log]
    accs_pct = [r['test_acc']*100 for r in log]
    cm       = np.array(m['test_confusion_matrix'])
    n_cls    = cm.shape[0]
    cls_lbl  = ['box', 'circle', 'clean', 'fall', 'run', 'walk']

    def _save_buf(fig):
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=300, bbox_inches='tight')
        buf.seek(0)
        plt.show()
        plt.close(fig)
        return buf

    # ── Training curve ────────────────────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(10, 5))
    ax1r = ax1.twinx()
    ln1  = ax1.plot( epochs, losses,   color='#E74C3C', linewidth=2, label='Loss')
    ln2  = ax1r.plot(epochs, accs_pct, color='#2ECC71', linewidth=2, label='Test Acc (%)')
    ax1.set_ylabel('Loss',          color='#E74C3C'); ax1.tick_params( axis='y', labelcolor='#E74C3C')
    ax1r.set_ylabel('Test Acc (%)', color='#2ECC71'); ax1r.tick_params(axis='y', labelcolor='#2ECC71')
    ax1r.set_ylim(0, 105)
    ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
    ax1.set_title('TF-Mamba NTU-Fi — Training Curve: Loss & Accuracy')
    lns = ln1 + ln2; ax1.legend(lns, [l.get_label() for l in lns], loc='center right')
    fig1.tight_layout()
    buf1 = _save_buf(fig1)

    # ── Confusion Matrix ──────────────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    cm_n = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    im   = ax2.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
    ax2.set_xticks(range(n_cls)); ax2.set_xticklabels(cls_lbl, fontsize=9, rotation=30)
    ax2.set_yticks(range(n_cls)); ax2.set_yticklabels(cls_lbl, fontsize=9)
    ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
    ax2.set_title('TF-Mamba NTU-Fi — Confusion Matrix (normalized)')
    fig2.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    for i in range(n_cls):
        for j in range(n_cls):
            ax2.text(j, i, f'{cm_n[i,j]:.2f}', ha='center', va='center', fontsize=8,
                     color='white' if cm_n[i, j] > 0.5 else 'black')
    fig2.tight_layout()
    buf2 = _save_buf(fig2)

    # ── Save ZIP ───────────────────────────────────────────────────────────────
    FOLDER   = 'tf_mamba_ntufi'
    zip_path = OUTPUT_DIR / 'results_summary.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr(f'{FOLDER}/plots/training_curve.png',   buf1.getvalue())
        zf.writestr(f'{FOLDER}/plots/confusion_matrix.png', buf2.getvalue())
        with open(_mpath, 'rb') as f:
            zf.writestr(f'{FOLDER}/metrics.json', f.read())
        with open(_lpath, 'rb') as f:
            zf.writestr(f'{FOLDER}/training_log.csv', f.read())
        _pt  = OUTPUT_DIR / 'final_model.pt'
        _npz = OUTPUT_DIR / 'test_predictions.npz'
        if _pt.exists():
            with open(_pt, 'rb') as f:
                zf.writestr(f'{FOLDER}/final_model.pt', f.read())
        if _npz.exists():
            with open(_npz, 'rb') as f:
                zf.writestr(f'{FOLDER}/test_predictions.npz', f.read())
    print(f'Saved ZIP: {zip_path}')